In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": "dataset/train.csv",
        "test": "dataset/test.csv",
    },
    encoding="cp1252",
)

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'],
        num_rows: 27481
    })
    test: Dataset({
        features: ['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'],
        num_rows: 4815
    })
})
{'textID': 'cb774db0d1', 'text': ' I`d have responded, if I were going', 'selected_text': 'I`d have responded, if I were going', 'sentiment': 'neutral', 'Time of Tweet': 'morning', 'Age of User': '0-20', 'Country': 'Afghanistan', 'Population -2020': 38928346, 'Land Area (Km²)': 652860.0, 'Density (P/Km²)': 60}


In [ ]:
from datasets import Value

columns_to_keep = ["textID", "text", "sentiment"]
label_map = {"negative": 0, "neutral": 1, "positive": 2}

columns_to_remove = [
    column
    for column in dataset["train"].column_names
    if column not in columns_to_keep
]

if columns_to_remove:
    dataset = dataset.remove_columns(columns_to_remove)


def encode_sentiment(example):
    sentiment = example["sentiment"]
    if sentiment is None:
        return {"sentiment": None}
    if isinstance(sentiment, int):
        return {"sentiment": sentiment}

    normalized = str(sentiment).strip().lower()
    if normalized.isdigit():
        return {"sentiment": int(normalized)}

    return {"sentiment": label_map[normalized]}


dataset = dataset.map(encode_sentiment)
dataset = dataset.cast_column("sentiment", Value("int64"))

print(dataset)
print(dataset["train"][0])
print(dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 27481
    })
    test: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 4815
    })
})
{'textID': 'cb774db0d1', 'text': ' I`d have responded, if I were going', 'sentiment': 1}
{'textID': 'f87dea47db', 'text': 'Last session of the day  http://twitpic.com/67ezh', 'sentiment': 1}


In [7]:
def get_invalid_reason(example):
    """Return reason if example is invalid; otherwise return None."""
    text = example.get("text", "")
    if text is None or (isinstance(text, str) and text.strip() == ""):
        return "empty_text"

    if example.get("sentiment") is None:
        return "missing_sentiment"

    if example.get("textID") is None:
        return "missing_textID"

    return None


def has_value(example):
    return get_invalid_reason(example) is None


filtered_out_examples = {}
for split in ["train", "test"]:
    dropped = []
    for idx, example in enumerate(dataset[split]):
        reason = get_invalid_reason(example)
        if reason is not None:
            dropped.append(
                {
                    "split": split,
                    "row_index": idx,
                    "textID": example.get("textID"),
                    "reason": reason,
                    "text": example.get("text"),
                }
            )
    filtered_out_examples[split] = dropped

# Filter out examples with no value
dataset["train"] = dataset["train"].filter(has_value)
dataset["test"] = dataset["test"].filter(has_value)

print("Dataset after removing examples with no value:")
print(dataset)
print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")

print("\nFiltered out examples summary:")
for split in ["train", "test"]:
    print(f"{split}: {len(filtered_out_examples[split])} rows removed")

print("\nFirst 5 filtered rows per split:")
for split in ["train", "test"]:
    print(f"\n{split}:")
    preview = filtered_out_examples[split][:5]
    if not preview:
        print("No rows removed")
    else:
        for item in preview:
            print(item)

Dataset after removing examples with no value:
DatasetDict({
    train: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 27480
    })
    test: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 3534
    })
})
Train set size: 27480
Test set size: 3534

Filtered out examples summary:
train: 1 rows removed
test: 1281 rows removed

First 5 filtered rows per split:

train:
{'split': 'train', 'row_index': 314, 'textID': 'fdb77c3752', 'reason': 'empty_text', 'text': None}

test:
{'split': 'test', 'row_index': 3534, 'textID': None, 'reason': 'empty_text', 'text': None}
{'split': 'test', 'row_index': 3535, 'textID': None, 'reason': 'empty_text', 'text': None}
{'split': 'test', 'row_index': 3536, 'textID': None, 'reason': 'empty_text', 'text': None}
{'split': 'test', 'row_index': 3537, 'textID': None, 'reason': 'empty_text', 'text': None}
{'split': 'test', 'row_index': 3538, 'textID': None, 'reason': 'empty_text', 'text': None}


### Train Dataset

Removed row 316 due to empty "text" and "selected_text" column

### Test Dataset

All rows contained data. Not sure why original dataset said there were 4815 rows, only found 3534 in the csv file.

In [8]:
import csv
import os

# Create output directory if it doesn't exist
output_dir = os.path.abspath("dataset/preprocessed_dataset")
os.makedirs(output_dir, exist_ok=True)

# Save the full dataset object (safe on Windows if target is in use)
try:
    dataset.save_to_disk(output_dir)
    print(f"Dataset saved to {output_dir}")
except OSError as e:
    print(f"Warning: save_to_disk failed: {e}")
    print("Continuing to save CSV outputs and filtered-out logs.")

# Also save individual splits as CSV for easy inspection
dataset["train"].to_csv(f"{output_dir}/train.csv", index=False)
dataset["test"].to_csv(f"{output_dir}/test.csv", index=False)
print(f"Train and test CSV files saved to {output_dir}")

# Save filtered-out examples logs
for split in ["train", "test"]:
    log_path = f"{output_dir}/{split}_filtered_out.csv"
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["split", "row_index", "textID", "reason", "text"]
        )
        writer.writeheader()
        writer.writerows(filtered_out_examples[split])
    print(f"Filtered-out log saved: {log_path}")

print(f"\nFinal dataset statistics:")
print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

Saving the dataset (0/1 shards):   0%|          | 0/27480 [00:00<?, ? examples/s]


Continuing to save CSV outputs and filtered-out logs.


Creating CSV from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 265.12ba/s]

Train and test CSV files saved to c:\repos\cs4248-tutorials\project\dataset\preprocessed_dataset
Filtered-out log saved: c:\repos\cs4248-tutorials\project\dataset\preprocessed_dataset/train_filtered_out.csv
Filtered-out log saved: c:\repos\cs4248-tutorials\project\dataset\preprocessed_dataset/test_filtered_out.csv

Final dataset statistics:
Train samples: 27480
Test samples: 3534


In [ ]:
from datasets import concatenate_datasets, DatasetDict

# Build a small, reproducible dataset for quick experiments
TOTAL_EXAMPLES = 500
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

# Combine cleaned splits, shuffle, and keep only the first 500 rows
combined = concatenate_datasets([dataset["train"], dataset["test"]]).shuffle(seed=SEED)
if len(combined) < TOTAL_EXAMPLES:
    raise ValueError(
        f"Not enough examples to build mini dataset: required {TOTAL_EXAMPLES}, found {len(combined)}"
    )

mini_all = combined.select(range(TOTAL_EXAMPLES))

# Exact split sizes: 350 train, 75 val, 75 test
train_size = int(TOTAL_EXAMPLES * TRAIN_RATIO)
val_size = int(TOTAL_EXAMPLES * VAL_RATIO)
test_size = TOTAL_EXAMPLES - train_size - val_size

mini_train = mini_all.select(range(0, train_size))
mini_val = mini_all.select(range(train_size, train_size + val_size))
mini_test = mini_all.select(range(train_size + val_size, TOTAL_EXAMPLES))

mini_dataset = DatasetDict(
    {
        "train": mini_train,
        "val": mini_val,
        "test": mini_test,
    }
)

mini_output_dir = os.path.abspath("dataset/mini_dataset")
os.makedirs(mini_output_dir, exist_ok=True)

# Save in Hugging Face dataset format and as CSV files
mini_dataset.save_to_disk(mini_output_dir)
mini_dataset["train"].to_csv(f"{mini_output_dir}/train.csv", index=False)
mini_dataset["val"].to_csv(f"{mini_output_dir}/val.csv", index=False)
mini_dataset["test"].to_csv(f"{mini_output_dir}/test.csv", index=False)

print("Mini dataset created and saved")
print(mini_dataset)
print(f"Total examples: {len(mini_dataset['train']) + len(mini_dataset['val']) + len(mini_dataset['test'])}")
print(f"Train: {len(mini_dataset['train'])}")
print(f"Val: {len(mini_dataset['val'])}")
print(f"Test: {len(mini_dataset['test'])}")
print(f"Saved to: {mini_output_dir}")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1002.22ba/s]

Mini dataset created and saved
DatasetDict({
    train: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 350
    })
    val: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 75
    })
    test: Dataset({
        features: ['textID', 'text', 'sentiment'],
        num_rows: 75
    })
})
Total examples: 500
Train: 350
Val: 75
Test: 75
Saved to: c:\repos\cs4248-tutorials\project\dataset\mini_dataset


: 

In [3]:
import os
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": "dataset/train.csv",
        "test": "dataset/test.csv",
    },
    encoding="cp1252",
)

print(dataset)
print(dataset["train"][0])

from datasets import DatasetDict

# Keep only text columns
columns_to_keep = ["textID", "text"]

text_only_splits = {}
for split in dataset.keys():
    cols_to_remove = [c for c in dataset[split].column_names if c not in columns_to_keep]
    text_only_splits[split] = dataset[split].remove_columns(cols_to_remove)

text_only_dataset = DatasetDict(text_only_splits)

# Split only the train split into train and val
train_val = text_only_dataset["train"].train_test_split(test_size=0.1, seed=42)

final_dataset = DatasetDict(
    {
        "train": train_val["train"],
        "val": train_val["test"],
        "test": text_only_dataset["test"],
    }
)

print(final_dataset)
print(f"Train size: {len(final_dataset['train'])}")
print(f"Val size: {len(final_dataset['val'])}")
print(f"Test size: {len(final_dataset['test'])}")
print("Sample row:", final_dataset["train"][0])

final_dataset.save_to_disk(os.path.abspath("dataset/text_only"))

DatasetDict({
    train: Dataset({
        features: ['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'],
        num_rows: 27481
    })
    test: Dataset({
        features: ['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'],
        num_rows: 4815
    })
})
{'textID': 'cb774db0d1', 'text': ' I`d have responded, if I were going', 'selected_text': 'I`d have responded, if I were going', 'sentiment': 'neutral', 'Time of Tweet': 'morning', 'Age of User': '0-20', 'Country': 'Afghanistan', 'Population -2020': 38928346, 'Land Area (Km²)': 652860.0, 'Density (P/Km²)': 60}
DatasetDict({
    train: Dataset({
        features: ['textID', 'text'],
        num_rows: 24732
    })
    val: Dataset({
        features: ['textID', 'text'],
        num_rows: 2749
    })
    test: Dataset({
        features: ['

Saving the dataset (1/1 shards): 100%|██████████| 4815/4815 [00:00<00:00, 1371609.19 examples/s]


In [2]:
import os
from datasets import concatenate_datasets, load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": "dataset/train.csv",
        "test": "dataset/test.csv",
    },
    encoding="cp1252",
)

columns_to_keep = ["textID", "text", "sentiment"]

train_trimmed = dataset["train"].remove_columns(
    [c for c in dataset["train"].column_names if c not in columns_to_keep]
)
test_trimmed = dataset["test"].remove_columns(
    [c for c in dataset["test"].column_names if c not in columns_to_keep]
)

full_dataset = concatenate_datasets([train_trimmed, test_trimmed])

# Drop rows where any kept column is missing or empty.
def has_no_empty_cells(example):
    for col in columns_to_keep:
        value = example.get(col)
        if value is None:
            return False
        if isinstance(value, str) and value.strip() == "":
            return False
    return True

before_rows = len(full_dataset)
full_dataset = full_dataset.filter(has_no_empty_cells)
after_rows = len(full_dataset)

output_dir = os.path.abspath("dataset/full_text_and_sentiment")
os.makedirs(output_dir, exist_ok=True)

full_dataset.save_to_disk(output_dir)
full_dataset.to_csv(f"{output_dir}/full.csv", index=False)

print(full_dataset)
print(f"Rows before dropping empty cells: {before_rows}")
print(f"Rows after dropping empty cells: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")
print("Sample row:", full_dataset[0])
print(f"Saved to: {output_dir}")

Filter:   0%|          | 0/32296 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/31014 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/32 [00:00<?, ?ba/s]

Dataset({
    features: ['textID', 'text', 'sentiment'],
    num_rows: 31014
})
Rows before dropping empty cells: 32296
Rows after dropping empty cells: 31014
Rows removed: 1282
Sample row: {'textID': 'cb774db0d1', 'text': ' I`d have responded, if I were going', 'sentiment': 'neutral'}
Saved to: c:\repos\cs4248-tutorials\project\dataset\full_text_and_sentiment
